# Experiments

This notebook studies the interaction between RM3 query expansion and TAS-B neural re-ranking on the MS MARCO Passage dataset.

This version is structured for faster iteration:
- reusable pipelines live in `src/pipelines.py`
- resultsets are cached to disk
- evaluation can be compared per query as well as in aggregate


## Setup

Import PyTerrier and the shared pipeline helpers.


In [1]:
from pathlib import Path

import pandas as pd
import pyterrier as pt

from src.analysis import (
    add_query_features,
    best_system_per_query,
    compare_query_sets,
    label_gain,
    summarize_best_systems,
    summarize_labels,
    top_queries,
)

from src.pipelines import (
    add_gain_columns,
    build_bm25,
    build_bm25_rm3,
    build_bm25_rm3_tasb,
    build_bm25_tasb,
    evaluate_runs,
    init_pyterrier,
    perquery_comparison,
    run_and_cache,
    sanitize_topics,
    subset_topics,
)

init_pyterrier()
CACHE_DIR = Path("results/cache")
CACHE_DIR.mkdir(parents=True, exist_ok=True)


Java started and loaded: pyterrier.java.colab, pyterrier.java, pyterrier.java.24, pyterrier.terrier.java [version=5.11 (build: craig.macdonald 2025-01-13 21:29), helper_version=0.0.8]


## Load dataset

Load the MS MARCO Passage dataset and the TREC DL 2019/2020 topic and qrel sets.


In [2]:
dataset = pt.get_dataset("msmarco_passage")
irds_dataset = pt.get_dataset("irds:msmarco-passage")

dl19 = pt.get_dataset("irds:msmarco-passage/trec-dl-2019/judged")
dl20 = pt.get_dataset("irds:msmarco-passage/trec-dl-2020/judged")

topics_19 = sanitize_topics(dl19.get_topics())
qrels_19 = dl19.get_qrels()

topics_20 = sanitize_topics(dl20.get_topics())
qrels_20 = dl20.get_qrels()

print(f"DL 2019 - topics: {len(topics_19)}, qrels: {len(qrels_19)}")
print(f"DL 2020 - topics: {len(topics_20)}, qrels: {len(qrels_20)}")


DL 2019 - topics: 43, qrels: 9260
DL 2020 - topics: 54, qrels: 11386


## Load index

Load the pre-built Terrier index for MS MARCO Passage.


In [3]:
index = pt.IndexFactory.of(dataset.get_index(variant="terrier_stemmed"))
print(index.getCollectionStatistics().toString())


Number of documents: 8841823
Number of terms: 1170682
Number of postings: 215238456
Number of fields: 1
Number of tokens: 288759529
Field names: [text]
Positions:   false



## Run configuration

Use `FAST_MODE` while developing. Switch it off for the final run.


In [4]:
FAST_MODE = False
CACHE_PREFIX = "fast" if FAST_MODE else "full"
NUM_RESULTS = 200 if FAST_MODE else 1000
RERANK_K = 30 if FAST_MODE else 100
FB_TERMS = 10
FB_VALUES = [10, 30] if FAST_MODE else [5, 10, 20, 30, 50]
N_TOPICS = 10 if FAST_MODE else None

topics_19_run, qrels_19_run = subset_topics(topics_19, qrels_19, n_topics=N_TOPICS)
topics_20_run, qrels_20_run = subset_topics(topics_20, qrels_20, n_topics=N_TOPICS)

print(f"FAST_MODE={FAST_MODE}")
print(f"CACHE_PREFIX={CACHE_PREFIX}")
print(f"DL19 run topics: {len(topics_19_run)}")
print(f"DL20 run topics: {len(topics_20_run)}")


FAST_MODE=False
CACHE_PREFIX=full
DL19 run topics: 43
DL20 run topics: 54


## Build pipelines

Create the four core retrieval systems.


In [5]:
bm25 = build_bm25(index, num_results=NUM_RESULTS)
bm25_rm3 = build_bm25_rm3(index, num_results=NUM_RESULTS, fb_terms=FB_TERMS)
bm25_tasb = build_bm25_tasb(index, irds_dataset, num_results=NUM_RESULTS, rerank_k=RERANK_K)
bm25_rm3_tasb = build_bm25_rm3_tasb(
    index,
    irds_dataset,
    num_results=NUM_RESULTS,
    fb_terms=FB_TERMS,
    rerank_k=RERANK_K,
)


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

## Cache DL19 runs

Run each pipeline once and save the ranking results to disk.


In [6]:
runs_19 = {
    "BM25": run_and_cache(bm25, topics_19_run, CACHE_DIR / f"{CACHE_PREFIX}_bm25_dl19.pkl"),
    "BM25+RM3": run_and_cache(
        bm25_rm3,
        topics_19_run,
        CACHE_DIR / f"{CACHE_PREFIX}_bm25_rm3_fb{FB_TERMS}_dl19.pkl",
    ),
    "BM25+TAS-B": run_and_cache(
        bm25_tasb,
        topics_19_run,
        CACHE_DIR / f"{CACHE_PREFIX}_bm25_tasb_k{RERANK_K}_dl19.pkl",
    ),
    "BM25+RM3+TAS-B": run_and_cache(
        bm25_rm3_tasb,
        topics_19_run,
        CACHE_DIR / f"{CACHE_PREFIX}_bm25_rm3_fb{FB_TERMS}_tasb_k{RERANK_K}_dl19.pkl",
    ),
}

{k: len(v) for k, v in runs_19.items()}


{'BM25': 42005, 'BM25+RM3': 43000, 'BM25+TAS-B': 4205, 'BM25+RM3+TAS-B': 4300}

## Evaluate DL19

Evaluate the cached resultsets without rerunning the pipelines.


In [7]:
evaluate_runs(runs_19, topics_19_run, qrels_19_run)

,name,map,recall_100,ndcg_cut_10
0,BM25,0.370004,0.442170,0.479540
1,BM25+RM3,0.406129,0.456336,0.524715
2,BM25+TAS-B,0.359250,0.442210,0.689030
3,BM25+RM3+TAS-B,0.367400,0.456336,0.687443


## Per-query comparison on DL19

Create a table with one row per query so systems can be compared topic by topic.


In [8]:
perquery_19 = perquery_comparison(runs_19, topics_19_run, qrels_19_run)
perquery_19 = add_gain_columns(
    perquery_19,
    baseline="BM25",
    systems=["BM25+RM3", "BM25+TAS-B", "BM25+RM3+TAS-B"],
)
perquery_19.sort_values("BM25+RM3+TAS-B_gain", ascending=False).head(10)


,qid,query,BM25,BM25+RM3,BM25+RM3+TAS-B,BM25+TAS-B,BM25+RM3_gain,BM25+TAS-B_gain,BM25+RM3+TAS-B_gain
25,183378,exons definition biology,0.366266,0.697505,1.000000,1.000000,0.331239,0.633734,0.633734
11,264014,how long is life cycle of flea,0.334922,0.355459,0.835440,0.845763,0.020536,0.510841,0.500518
18,1115776,what is an aml surveillance analyst,0.334768,0.361401,0.834189,0.766642,0.026633,0.431873,0.499420
35,1114646,what is famvir prescribed for,0.363148,0.358963,0.815633,0.815633,-0.004185,0.452485,0.452485
15,148538,difference between rn and bsn,0.163883,0.263902,0.590363,0.617333,0.100020,0.453450,0.426480
28,490595,rsa definition key,0.223715,0.190264,0.634853,0.634853,-0.033450,0.411138,0.411138
41,1129237,hydrogen is a liquid below what temperature,0.547601,0.656812,0.929698,0.810827,0.109211,0.263225,0.382096
31,443396,lps laws definition,0.069431,0.066254,0.435098,0.384311,-0.003177,0.314880,0.365666
38,405717,is cdg airport in main paris,0.375915,0.340584,0.738404,0.738404,-0.035331,0.362488,0.362488
13,962179,when was the salvation army founded,0.000000,0.000000,0.358519,0.570542,0.000000,0.570542,0.358519


## Inspect a single query

Filter to one query/topic to inspect its metrics more closely.


In [9]:
example_qid = str(perquery_19.iloc[0]["qid"])
perquery_19[perquery_19["qid"].astype(str) == example_qid]


,qid,query,BM25,BM25+RM3,BM25+RM3+TAS-B,BM25+TAS-B,BM25+RM3_gain,BM25+TAS-B_gain,BM25+RM3+TAS-B_gain
0,156493,do goldfish grow,0.963412,1.0,0.900864,0.900864,0.036588,-0.062548,-0.062548


## Cache and evaluate DL20

Repeat the same workflow for TREC DL 2020.


In [10]:
runs_20 = {
    "BM25": run_and_cache(bm25, topics_20_run, CACHE_DIR / f"{CACHE_PREFIX}_bm25_dl20.pkl"),
    "BM25+RM3": run_and_cache(
        bm25_rm3,
        topics_20_run,
        CACHE_DIR / f"{CACHE_PREFIX}_bm25_rm3_fb{FB_TERMS}_dl20.pkl",
    ),
    "BM25+TAS-B": run_and_cache(
        bm25_tasb,
        topics_20_run,
        CACHE_DIR / f"{CACHE_PREFIX}_bm25_tasb_k{RERANK_K}_dl20.pkl",
    ),
    "BM25+RM3+TAS-B": run_and_cache(
        bm25_rm3_tasb,
        topics_20_run,
        CACHE_DIR / f"{CACHE_PREFIX}_bm25_rm3_fb{FB_TERMS}_tasb_k{RERANK_K}_dl20.pkl",
    ),
}

evaluate_runs(runs_20, topics_20_run, qrels_20_run)


,name,map,recall_100,ndcg_cut_10
0,BM25,0.358724,0.504658,0.493627
1,BM25+RM3,0.400468,0.548754,0.510773
2,BM25+TAS-B,0.385519,0.505561,0.657714
3,BM25+RM3+TAS-B,0.412778,0.549097,0.671603


## Per-query comparison on DL20


In [11]:
perquery_20 = perquery_comparison(runs_20, topics_20_run, qrels_20_run)
perquery_20 = add_gain_columns(
    perquery_20,
    baseline="BM25",
    systems=["BM25+RM3", "BM25+TAS-B", "BM25+RM3+TAS-B"],
)
perquery_20.sort_values("BM25+RM3+TAS-B_gain", ascending=False).head(10)


,qid,query,BM25,BM25+RM3,BM25+RM3+TAS-B,BM25+TAS-B,BM25+RM3_gain,BM25+TAS-B_gain,BM25+RM3+TAS-B_gain
32,324585,how much money do motivational speakers make,0.069431,0.069431,0.889954,0.889954,0.000000,0.820523,0.820523
41,583468,what carvedilol used for,0.275930,0.257691,0.936808,0.936808,-0.018239,0.660877,0.660877
18,1132532,average annual income data analyst,0.150654,0.176413,0.738222,0.734567,0.025759,0.583913,0.587568
51,938400,when did family feud come out?,0.114339,0.134713,0.655737,0.655737,0.020374,0.541398,0.541398
17,1131069,how many sons robert kraft has,0.469279,0.758587,1.000000,1.000000,0.289308,0.530721,0.530721
53,997622,where is the show shameless filmed,0.481900,0.710023,1.000000,0.955831,0.228123,0.473931,0.518100
52,940547,when did rock n roll begin?,0.127349,0.127349,0.611993,0.547782,0.000000,0.420433,0.484643
35,336901,how old is vanessa redgrave,0.156426,0.167160,0.625705,0.617320,0.010734,0.460893,0.469279
10,1110678,what is the un fao,0.332921,0.433885,0.747478,0.666948,0.100964,0.334027,0.414557
1,1037496,who is rep scalise?,0.542564,0.522524,0.927449,0.885036,-0.020039,0.342472,0.384886


## RM3 parameter study with caching on DL 19

Only compute each `fb_terms` setting once, then compare the saved outputs.


In [12]:
rm3_tasb_runs_19 = {"BM25+TAS-B": runs_19["BM25+TAS-B"]}

for fb in FB_VALUES:
    pipe = build_bm25_rm3_tasb(
        index,
        irds_dataset,
        num_results=NUM_RESULTS,
        fb_terms=fb,
        rerank_k=RERANK_K,
    )
    rm3_tasb_runs_19[f"BM25+RM3(fb={fb})+TAS-B"] = run_and_cache(
        pipe,
        topics_19_run,
        CACHE_DIR / f"{CACHE_PREFIX}_bm25_rm3_fb{fb}_tasb_k{RERANK_K}_dl19.pkl",
    )

evaluate_runs(rm3_tasb_runs_19, topics_19_run, qrels_19_run)


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

,name,map,recall_100,ndcg_cut_10
0,BM25+TAS-B,0.359250,0.442210,0.689030
1,BM25+RM3(fb=5)+TAS-B,0.364385,0.454594,0.686137
2,BM25+RM3(fb=10)+TAS-B,0.367400,0.456336,0.687443
3,BM25+RM3(fb=20)+TAS-B,0.372978,0.465349,0.688454
4,BM25+RM3(fb=30)+TAS-B,0.372421,0.464965,0.688710
5,BM25+RM3(fb=50)+TAS-B,0.373623,0.465664,0.688618


## Per-query RM3 study on DL19

Compare which queries benefit from more expansion terms.


In [13]:
rm3_perquery_19 = perquery_comparison(
    rm3_tasb_runs_19,
    topics_19_run,
    qrels_19_run,
)
rm3_perquery_19.head(10)


,qid,query,BM25+RM3(fb=10)+TAS-B,BM25+RM3(fb=20)+TAS-B,BM25+RM3(fb=30)+TAS-B,BM25+RM3(fb=5)+TAS-B,BM25+RM3(fb=50)+TAS-B,BM25+TAS-B
0,156493,do goldfish grow,0.900864,0.900864,0.900864,0.900864,0.900864,0.900864
1,1110199,what is wifi vs bluetooth,0.412420,0.412420,0.412420,0.412420,0.412420,0.412420
2,1063750,why did the us volunterilay enter ww1,0.379911,0.379911,0.379911,0.379911,0.379911,0.379911
3,130510,definition declaratory judgment,0.964564,0.964564,0.964564,0.964564,0.964564,0.964564
4,489204,right pelvic pain causes,0.380219,0.380219,0.380219,0.380219,0.380219,0.380219
5,573724,what are the social determinants of health,0.663456,0.663456,0.663456,0.663456,0.663456,0.663456
6,168216,does legionella pneumophila cause pneumonia,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
7,1133167,how is the weather in jamaica,0.698263,0.698263,0.698263,0.698263,0.698263,0.695048
8,527433,types of dysarthria from cerebral palsy,0.620995,0.656124,0.656124,0.630389,0.656124,0.703943
9,1037798,who is robert gray,0.204275,0.204275,0.204275,0.204275,0.204275,0.204275


## Analyze RM3 effects on DL19

Use the cached per-query table to see which queries benefit from RM3 before TAS-B and which ones get worse.


In [14]:
rm3_analysis_19 = add_gain_columns(
    rm3_perquery_19,
    baseline="BM25+TAS-B",
    systems=[f"BM25+RM3(fb={fb})+TAS-B" for fb in FB_VALUES],
)
rm3_analysis_19 = add_query_features(rm3_analysis_19)
rm3_analysis_19 = label_gain(
    rm3_analysis_19,
    gain_column=f"BM25+RM3(fb={FB_TERMS})+TAS-B_gain",
)

summarize_labels(rm3_analysis_19, f"BM25+RM3(fb={FB_TERMS})+TAS-B_gain_label")


,BM25+RM3(fb=10)+TAS-B_gain_label,count
0,neutral,27
1,hurt,9
2,helped,7


In [15]:
top_queries(
    rm3_analysis_19,
    f"BM25+RM3(fb={FB_TERMS})+TAS-B_gain",
    n=10,
    ascending=False,
)

,qid,query,BM25+RM3(fb=10)+TAS-B_gain,BM25+RM3(fb=10)+TAS-B,BM25+RM3(fb=20)+TAS-B,BM25+RM3(fb=30)+TAS-B,BM25+RM3(fb=5)+TAS-B,BM25+RM3(fb=50)+TAS-B,BM25+TAS-B,BM25+RM3(fb=5)+TAS-B_gain,BM25+RM3(fb=20)+TAS-B_gain,BM25+RM3(fb=30)+TAS-B_gain,BM25+RM3(fb=50)+TAS-B_gain,query_len_words,query_len_chars,has_colon,has_digit,is_question
41,1129237,hydrogen is a liquid below what temperature,0.118871,0.929698,0.929698,0.929698,0.929698,0.929698,0.810827,0.118871,0.118871,0.118871,0.118871,7,43,False,False,False
36,19335,anthropological definition of environment,0.095534,0.596099,0.596099,0.596099,0.565071,0.596099,0.500565,0.064506,0.095534,0.095534,0.095534,4,41,False,False,False
19,1112341,what is the daily life of thai people,0.088520,0.824226,0.803019,0.803019,0.824226,0.803019,0.735705,0.088520,0.067313,0.067313,0.067313,8,37,False,False,False
18,1115776,what is an aml surveillance analyst,0.067547,0.834189,0.766642,0.766642,0.801100,0.766642,0.766642,0.034458,0.000000,0.000000,0.000000,6,35,False,False,False
10,915593,what types of food can you cook sous vide,0.066619,0.510716,0.600470,0.600470,0.444097,0.600470,0.444097,0.000000,0.156373,0.156373,0.156373,9,41,False,False,False
31,443396,lps laws definition,0.050786,0.435098,0.412692,0.388244,0.435098,0.384311,0.384311,0.050786,0.028381,0.003933,0.000000,3,19,False,False,False
23,207786,how are some sharks warm blooded,0.036461,0.595740,0.595740,0.595740,0.595740,0.595740,0.559279,0.036461,0.036461,0.036461,0.036461,6,32,False,False,False
7,1133167,how is the weather in jamaica,0.003215,0.698263,0.698263,0.698263,0.698263,0.698263,0.695048,0.003215,0.003215,0.003215,0.003215,6,29,False,False,False
22,833860,what is the most popular food in switzerland,0.001678,0.938863,0.945260,0.945260,0.945260,0.945260,0.937185,0.008075,0.008075,0.008075,0.008075,8,44,False,False,False
32,1121709,what are the three percenters?,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,5,30,False,False,True


In [16]:
top_queries(
    rm3_analysis_19,
    f"BM25+RM3(fb={FB_TERMS})+TAS-B_gain",
    n=10,
    ascending=True,
)

,qid,query,BM25+RM3(fb=10)+TAS-B_gain,BM25+RM3(fb=10)+TAS-B,BM25+RM3(fb=20)+TAS-B,BM25+RM3(fb=30)+TAS-B,BM25+RM3(fb=5)+TAS-B,BM25+RM3(fb=50)+TAS-B,BM25+TAS-B,BM25+RM3(fb=5)+TAS-B_gain,BM25+RM3(fb=20)+TAS-B_gain,BM25+RM3(fb=30)+TAS-B_gain,BM25+RM3(fb=50)+TAS-B_gain,query_len_words,query_len_chars,has_colon,has_digit,is_question
13,962179,when was the salvation army founded,-0.212023,0.358519,0.358519,0.358519,0.358519,0.358519,0.570542,-0.212023,-0.212023,-0.212023,-0.212023,6,35,False,False,False
26,1106007,define visceral?,-0.130569,0.626394,0.626394,0.626394,0.626394,0.626394,0.756963,-0.130569,-0.130569,-0.130569,-0.130569,2,16,False,False,True
8,527433,types of dysarthria from cerebral palsy,-0.082948,0.620995,0.656124,0.656124,0.630389,0.656124,0.703943,-0.073554,-0.047819,-0.047819,-0.047819,6,39,False,False,False
33,87452,causes of military suicide,-0.051388,0.488759,0.540147,0.540147,0.466674,0.540147,0.540147,-0.073472,0.000000,0.000000,0.000000,4,26,False,False,False
40,1113437,what is physical description of spruce,-0.036094,0.251215,0.198722,0.234191,0.332125,0.234191,0.287309,0.044816,-0.088587,-0.053118,-0.053118,6,38,False,False,False
15,148538,difference between rn and bsn,-0.026970,0.590363,0.590363,0.590363,0.590363,0.590363,0.617333,-0.026970,-0.026970,-0.026970,-0.026970,5,29,False,False,False
24,1114819,what is durable medical equipment consist of,-0.024455,0.788901,0.813356,0.813356,0.788901,0.813356,0.813356,-0.024455,0.000000,0.000000,0.000000,7,44,False,False,False
37,47923,axon terminals or synaptic knob definition,-0.022716,0.542444,0.542444,0.542444,0.542444,0.542444,0.565161,-0.022716,-0.022716,-0.022716,-0.022716,6,42,False,False,False
11,264014,how long is life cycle of flea,-0.010323,0.835440,0.835440,0.835440,0.835440,0.835440,0.845763,-0.010323,-0.010323,-0.010323,-0.010323,7,30,False,False,False
27,1124210,tracheids are part of _____.,0.000000,0.784233,0.784233,0.784233,0.784233,0.784233,0.784233,0.000000,0.000000,0.000000,0.000000,5,28,False,False,False


In [17]:
compare_query_sets(
    rm3_analysis_19,
    label_column=f"BM25+RM3(fb={FB_TERMS})+TAS-B_gain_label",
    feature_columns=["query_len_words", "query_len_chars", "has_colon", "has_digit", "is_question"],
)


,BM25+RM3(fb=10)+TAS-B_gain_label,query_len_words,query_len_chars,has_colon,has_digit,is_question
0,helped,6.142857,35.428571,0.0,0.000000,0.000000
1,hurt,5.444444,33.222222,0.0,0.000000,0.111111
2,neutral,5.185185,31.888889,0.0,0.037037,0.037037


## Which fb\_terms works best on DL19?

Inspect both the average scores and which `fb_terms` wins most often per query.


In [34]:
rm3_metric_cols_19 = [f"BM25+RM3(fb={fb})+TAS-B" for fb in FB_VALUES]
rm3_gain_cols_19 = [f"BM25+RM3(fb={fb})+TAS-B_gain" for fb in FB_VALUES]

rm3_analysis_19[rm3_metric_cols_19].mean().sort_values(ascending=False)

BM25+RM3(fb=30)+TAS-B    0.688710
BM25+RM3(fb=50)+TAS-B    0.688618
BM25+RM3(fb=20)+TAS-B    0.688454
BM25+RM3(fb=10)+TAS-B    0.687443
BM25+RM3(fb=5)+TAS-B     0.686137
dtype: float64

In [35]:
rm3_best_19 = best_system_per_query(rm3_analysis_19, rm3_metric_cols_19)
summarize_best_systems(rm3_best_19)

,best_system,count
0,BM25+RM3(fb=5)+TAS-B,37
1,BM25+RM3(fb=20)+TAS-B,4
2,BM25+RM3(fb=10)+TAS-B,2


In [36]:
rm3_best_gain_19 = best_system_per_query(rm3_analysis_19, rm3_gain_cols_19, output_column="best_gain_setting")
summarize_best_systems(rm3_best_gain_19, best_system_column="best_gain_setting")

,best_gain_setting,count
0,BM25+RM3(fb=5)+TAS-B_gain,37
1,BM25+RM3(fb=20)+TAS-B_gain,4
2,BM25+RM3(fb=10)+TAS-B_gain,2


## Recall analysis per `fb_terms` on DL19

Compare `BM25` against `BM25+RM3(fb=k)` using `recall_100` to see which expansion setting retrieves the strongest candidate set.


In [53]:
bm25_rm3_runs_19 = {"BM25": runs_19["BM25"]}

for fb in FB_VALUES:
    pipe = build_bm25_rm3(index, num_results=NUM_RESULTS, fb_terms=fb)
    bm25_rm3_runs_19[f"BM25+RM3(fb={fb})"] = run_and_cache(
        pipe,
        topics_19_run,
        CACHE_DIR / f"{CACHE_PREFIX}_bm25_rm3_fb{fb}_dl19.pkl",
    )

bm25_rm3_perquery_19_recall = perquery_comparison(
    bm25_rm3_runs_19,
    topics_19_run,
    qrels_19_run,
    metric="recall_100",
)

bm25_rm3_analysis_19_recall = add_gain_columns(
    bm25_rm3_perquery_19_recall,
    baseline="BM25",
    systems=[f"BM25+RM3(fb={fb})" for fb in FB_VALUES],
)
bm25_rm3_analysis_19_recall = add_query_features(bm25_rm3_analysis_19_recall)


In [54]:
bm25_rm3_metric_cols_19 = [f"BM25+RM3(fb={fb})" for fb in FB_VALUES]
bm25_rm3_gain_cols_19 = [f"BM25+RM3(fb={fb})_gain" for fb in FB_VALUES]

bm25_rm3_analysis_19_recall[bm25_rm3_metric_cols_19].mean().sort_values(ascending=False)


BM25+RM3(fb=50)    0.465664
BM25+RM3(fb=20)    0.465349
BM25+RM3(fb=30)    0.464965
BM25+RM3(fb=10)    0.456336
BM25+RM3(fb=5)     0.454525
dtype: float64

In [55]:
bm25_rm3_best_19 = best_system_per_query(bm25_rm3_analysis_19_recall, bm25_rm3_metric_cols_19)
summarize_best_systems(bm25_rm3_best_19)


,best_system,count
0,BM25+RM3(fb=5),24
1,BM25+RM3(fb=20),11
2,BM25+RM3(fb=10),4
3,BM25+RM3(fb=50),3
4,BM25+RM3(fb=30),1


In [56]:
bm25_rm3_best_gain_19 = best_system_per_query(
    bm25_rm3_analysis_19_recall,
    bm25_rm3_gain_cols_19,
    output_column="best_gain_setting",
)
summarize_best_systems(bm25_rm3_best_gain_19, best_system_column="best_gain_setting")


,best_gain_setting,count
0,BM25+RM3(fb=5)_gain,24
1,BM25+RM3(fb=20)_gain,11
2,BM25+RM3(fb=10)_gain,4
3,BM25+RM3(fb=50)_gain,3
4,BM25+RM3(fb=30)_gain,1


In [57]:
bm25_rm3_analysis_19_recall = label_gain(
    bm25_rm3_analysis_19_recall,
    gain_column=f"BM25+RM3(fb={FB_TERMS})_gain",
)

summarize_labels(bm25_rm3_analysis_19_recall, f"BM25+RM3(fb={FB_TERMS})_gain_label")


,BM25+RM3(fb=10)_gain_label,count
0,neutral,19
1,helped,18
2,hurt,6


In [58]:
top_queries(
    bm25_rm3_analysis_19_recall,
    f"BM25+RM3(fb={FB_TERMS})_gain",
    n=10,
    ascending=False,
)


,qid,query,BM25+RM3(fb=10)_gain,BM25,BM25+RM3(fb=10),BM25+RM3(fb=20),BM25+RM3(fb=30),BM25+RM3(fb=5),BM25+RM3(fb=50),BM25+RM3(fb=5)_gain,BM25+RM3(fb=20)_gain,BM25+RM3(fb=30)_gain,BM25+RM3(fb=50)_gain,query_len_words,query_len_chars,has_colon,has_digit,is_question
25,183378,exons definition biology,0.113537,0.235808,0.349345,0.344978,0.349345,0.336245,0.353712,0.100437,0.109170,0.113537,0.117904,3,24,False,False,False
40,1113437,what is physical description of spruce,0.103896,0.272727,0.376623,0.324675,0.324675,0.363636,0.324675,0.090909,0.051948,0.051948,0.051948,6,38,False,False,False
29,1103812,who formed the commonwealth of independent states,0.096774,0.612903,0.709677,0.741935,0.741935,0.709677,0.741935,0.096774,0.129032,0.129032,0.129032,7,49,False,False,False
9,1037798,who is robert gray,0.076923,0.615385,0.692308,0.769231,0.769231,0.692308,0.769231,0.076923,0.153846,0.153846,0.153846,4,18,False,False,False
14,1117099,what is a active margin,0.075630,0.310924,0.386555,0.394958,0.386555,0.378151,0.386555,0.067227,0.084034,0.075630,0.075630,5,23,False,False,False
17,359349,how to find the midsegment of a trapezoid,0.071429,0.500000,0.571429,0.571429,0.589286,0.589286,0.589286,0.089286,0.071429,0.089286,0.089286,8,41,False,False,False
41,1129237,hydrogen is a liquid below what temperature,0.071429,0.678571,0.750000,0.750000,0.750000,0.750000,0.750000,0.071429,0.071429,0.071429,0.071429,7,43,False,False,False
35,1114646,what is famvir prescribed for,0.057692,0.673077,0.730769,0.730769,0.730769,0.750000,0.730769,0.076923,0.057692,0.057692,0.057692,5,29,False,False,False
22,833860,what is the most popular food in switzerland,0.053333,0.226667,0.280000,0.306667,0.280000,0.306667,0.280000,0.080000,0.080000,0.053333,0.053333,8,44,False,False,False
26,1106007,define visceral?,0.050000,0.233333,0.283333,0.266667,0.266667,0.266667,0.266667,0.033333,0.033333,0.033333,0.033333,2,16,False,False,True


## RM3 parameter study with caching on DL20

Repeat the RM3 parameter sweep on DL20 so the effect of `fb_terms` can be checked on both evaluation sets.


In [51]:
rm3_tasb_runs_20 = {"BM25+TAS-B": runs_20["BM25+TAS-B"]}

for fb in FB_VALUES:
    pipe = build_bm25_rm3_tasb(
        index,
        irds_dataset,
        num_results=NUM_RESULTS,
        fb_terms=fb,
        rerank_k=RERANK_K,
    )
    rm3_tasb_runs_20[f"BM25+RM3(fb={fb})+TAS-B"] = run_and_cache(
        pipe,
        topics_20_run,
        CACHE_DIR / f"{CACHE_PREFIX}_bm25_rm3_fb{fb}_tasb_k{RERANK_K}_dl20.pkl",
    )

evaluate_runs(rm3_tasb_runs_20, topics_20_run, qrels_20_run)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

,name,map,recall_100,ndcg_cut_10
0,BM25+TAS-B,0.385519,0.505561,0.657714
1,BM25+RM3(fb=5)+TAS-B,0.415240,0.552288,0.674037
2,BM25+RM3(fb=10)+TAS-B,0.412778,0.549097,0.671603
3,BM25+RM3(fb=20)+TAS-B,0.414257,0.549762,0.676079
4,BM25+RM3(fb=30)+TAS-B,0.415657,0.551131,0.676006
5,BM25+RM3(fb=50)+TAS-B,0.416002,0.551416,0.675926


## Per-query RM3 study on DL20

Inspect which DL20 queries benefit from different numbers of expansion terms.


In [22]:
rm3_perquery_20 = perquery_comparison(
    rm3_tasb_runs_20,
    topics_20_run,
    qrels_20_run,
)
rm3_perquery_20.head(10)

,qid,query,BM25+RM3(fb=10)+TAS-B,BM25+RM3(fb=20)+TAS-B,BM25+RM3(fb=30)+TAS-B,BM25+RM3(fb=5)+TAS-B,BM25+RM3(fb=50)+TAS-B,BM25+TAS-B
0,1030303,who is aziz hashim,0.786270,0.786270,0.786270,0.786270,0.786270,0.786270
1,1037496,who is rep scalise?,0.927449,0.927449,0.927449,0.927449,0.927449,0.885036
2,1043135,who killed nicholas ii of russia,0.505473,0.505473,0.505473,0.505473,0.543548,0.543548
3,1051399,who sings monk theme song,0.563687,0.674954,0.706707,0.563687,0.706707,0.797741
4,1064670,why do hunters pattern their shotguns?,0.873032,0.873032,0.873032,0.873032,0.873032,0.873032
5,1071750,why is pete rose banned from hall of fame,0.475741,0.475741,0.475741,0.475741,0.475741,0.475741
6,1105792,define geon,0.832729,0.832729,0.832729,0.832729,0.832729,0.832729
7,1106979,define pareto chart in statistics,0.839826,0.839826,0.839826,0.839826,0.839826,0.839826
8,1108651,what the best way to get clothes white,0.644030,0.686443,0.686443,0.665236,0.644030,0.430717
9,1109707,what medium do radio waves travel through,0.816010,0.816010,0.837217,0.837217,0.837217,0.811392


## Analyze RM3 effects on DL20

Repeat the same query-level analysis for DL20.


In [23]:
rm3_analysis_20 = add_gain_columns(
    rm3_perquery_20,
    baseline="BM25+TAS-B",
    systems=[f"BM25+RM3(fb={fb})+TAS-B" for fb in FB_VALUES],
)
rm3_analysis_20 = add_query_features(rm3_analysis_20)
rm3_analysis_20 = label_gain(
    rm3_analysis_20,
    gain_column=f"BM25+RM3(fb={FB_TERMS})+TAS-B_gain",
)

summarize_labels(rm3_analysis_20, f"BM25+RM3(fb={FB_TERMS})+TAS-B_gain_label")


,BM25+RM3(fb=10)+TAS-B_gain_label,count
0,neutral,41
1,helped,9
2,hurt,4


In [24]:
top_queries(
    rm3_analysis_20,
    f"BM25+RM3(fb={FB_TERMS})+TAS-B_gain",
    n=10,
    ascending=False,
)

,qid,query,BM25+RM3(fb=10)+TAS-B_gain,BM25+RM3(fb=10)+TAS-B,BM25+RM3(fb=20)+TAS-B,BM25+RM3(fb=30)+TAS-B,BM25+RM3(fb=5)+TAS-B,BM25+RM3(fb=50)+TAS-B,BM25+TAS-B,BM25+RM3(fb=5)+TAS-B_gain,BM25+RM3(fb=20)+TAS-B_gain,BM25+RM3(fb=30)+TAS-B_gain,BM25+RM3(fb=50)+TAS-B_gain,query_len_words,query_len_chars,has_colon,has_digit,is_question
14,1121353,what can you do about discrimination in the wo...,0.270007,0.825453,0.728026,0.823291,0.873232,0.823291,0.555447,0.317785,0.172579,0.267844,0.267844,12,70,False,False,False
29,174463,dog day afternoon meaning,0.261840,0.582539,0.582539,0.518043,0.582539,0.518043,0.320699,0.261840,0.261840,0.197344,0.197344,4,25,False,False,False
8,1108651,what the best way to get clothes white,0.213313,0.644030,0.686443,0.686443,0.665236,0.644030,0.430717,0.234520,0.255727,0.255727,0.213313,8,38,False,False,False
10,1110678,what is the un fao,0.080530,0.747478,0.747478,0.747478,0.747478,0.747478,0.666948,0.080530,0.080530,0.080530,0.080530,5,18,False,False,False
52,940547,when did rock n roll begin?,0.064210,0.611993,0.539587,0.539587,0.547782,0.539587,0.547782,0.000000,-0.008195,-0.008195,-0.008195,6,27,False,False,True
13,1116380,what is a nonconformity? earth science,0.051617,0.118177,0.118177,0.118177,0.154690,0.118177,0.066560,0.088130,0.051617,0.051617,0.051617,6,38,False,False,False
30,23849,are naturalization records public information,0.044983,0.432318,0.432318,0.432318,0.408021,0.432318,0.387335,0.020686,0.044983,0.044983,0.044983,5,45,False,False,False
53,997622,where is the show shameless filmed,0.044169,1.000000,1.000000,1.000000,1.000000,1.000000,0.955831,0.044169,0.044169,0.044169,0.044169,6,34,False,False,False
1,1037496,who is rep scalise?,0.042414,0.927449,0.927449,0.927449,0.927449,0.927449,0.885036,0.042414,0.042414,0.042414,0.042414,4,19,False,False,True
40,555530,what are best foods to lower cholesterol,0.009444,0.373260,0.373260,0.373260,0.338544,0.373260,0.363816,-0.025271,0.009444,0.009444,0.009444,7,40,False,False,False


In [25]:
top_queries(
    rm3_analysis_20,
    f"BM25+RM3(fb={FB_TERMS})+TAS-B_gain",
    n=10,
    ascending=True,
)

,qid,query,BM25+RM3(fb=10)+TAS-B_gain,BM25+RM3(fb=10)+TAS-B,BM25+RM3(fb=20)+TAS-B,BM25+RM3(fb=30)+TAS-B,BM25+RM3(fb=5)+TAS-B,BM25+RM3(fb=50)+TAS-B,BM25+TAS-B,BM25+RM3(fb=5)+TAS-B_gain,BM25+RM3(fb=20)+TAS-B_gain,BM25+RM3(fb=30)+TAS-B_gain,BM25+RM3(fb=50)+TAS-B_gain,query_len_words,query_len_chars,has_colon,has_digit,is_question
3,1051399,who sings monk theme song,-0.234054,0.563687,0.674954,0.706707,0.563687,0.706707,0.797741,-0.234054,-0.122786,-0.091033,-0.091033,5,25,False,False,False
42,640502,what does it mean if your tsh is low,-0.051279,0.863015,0.933746,0.933746,0.933746,0.933746,0.914294,0.019451,0.019451,0.019451,0.019451,9,36,False,False,False
2,1043135,who killed nicholas ii of russia,-0.038074,0.505473,0.505473,0.505473,0.505473,0.543548,0.543548,-0.038074,-0.038074,-0.038074,0.000000,6,32,False,False,False
43,67316,can fever cause miscarriage early pregnancy,-0.028745,0.293011,0.293011,0.293011,0.293011,0.293011,0.321756,-0.028745,-0.028745,-0.028745,-0.028745,6,43,False,False,False
28,169208,does mississippi have an income tax,0.000000,0.693266,0.693266,0.693266,0.693266,0.693266,0.693266,0.000000,0.000000,0.000000,0.000000,6,35,False,False,False
31,258062,how long does it take to remove wisdom tooth,0.000000,0.623554,0.692985,0.692985,0.692985,0.692985,0.623554,0.069431,0.069431,0.069431,0.069431,9,44,False,False,False
32,324585,how much money do motivational speakers make,0.000000,0.889954,0.889954,0.889954,0.889954,0.889954,0.889954,0.000000,0.000000,0.000000,0.000000,7,44,False,False,False
33,330975,how much would it cost to install my own wind ...,0.000000,0.621746,0.621746,0.621746,0.619628,0.621746,0.621746,-0.002118,0.000000,0.000000,0.000000,11,53,False,False,False
34,332593,how often to button quail lay eggs,0.000000,0.728769,0.728769,0.728769,0.728769,0.728769,0.728769,0.000000,0.000000,0.000000,0.000000,7,34,False,False,False
37,405163,is caffeine an narcotic,0.000000,0.420217,0.420217,0.420217,0.420217,0.420217,0.420217,0.000000,0.000000,0.000000,0.000000,4,23,False,False,False


In [26]:
compare_query_sets(
    rm3_analysis_20,
    label_column=f"BM25+RM3(fb={FB_TERMS})+TAS-B_gain_label",
    feature_columns=["query_len_words", "query_len_chars", "has_colon", "has_digit", "is_question"],
)

,BM25+RM3(fb=10)+TAS-B_gain_label,query_len_words,query_len_chars,has_colon,has_digit,is_question
0,helped,6.222222,34.888889,0.0,0.0,0.222222
1,hurt,6.500000,34.000000,0.0,0.0,0.000000
2,neutral,5.951220,33.512195,0.0,0.0,0.073171


## Which fb\_terms works best on DL20?

Use the same two views for DL20: average performance and best setting per query.


In [31]:
rm3_metric_cols_20 = [f"BM25+RM3(fb={fb})+TAS-B" for fb in FB_VALUES]
rm3_gain_cols_20 = [f"BM25+RM3(fb={fb})+TAS-B_gain" for fb in FB_VALUES]

rm3_analysis_20[rm3_metric_cols_20].mean().sort_values(ascending=False)

BM25+RM3(fb=20)+TAS-B    0.676079
BM25+RM3(fb=30)+TAS-B    0.676006
BM25+RM3(fb=50)+TAS-B    0.675926
BM25+RM3(fb=5)+TAS-B     0.674037
BM25+RM3(fb=10)+TAS-B    0.671603
dtype: float64

In [32]:
rm3_best_20 = best_system_per_query(rm3_analysis_20, rm3_metric_cols_20)
summarize_best_systems(rm3_best_20)

,best_system,count
0,BM25+RM3(fb=5)+TAS-B,44
1,BM25+RM3(fb=10)+TAS-B,5
2,BM25+RM3(fb=20)+TAS-B,3
3,BM25+RM3(fb=50)+TAS-B,1
4,BM25+RM3(fb=30)+TAS-B,1


In [33]:
rm3_best_gain_20 = best_system_per_query(rm3_analysis_20, rm3_gain_cols_20, output_column="best_gain_setting")
summarize_best_systems(rm3_best_gain_20, best_system_column="best_gain_setting")

,best_gain_setting,count
0,BM25+RM3(fb=5)+TAS-B_gain,44
1,BM25+RM3(fb=10)+TAS-B_gain,5
2,BM25+RM3(fb=20)+TAS-B_gain,3
3,BM25+RM3(fb=50)+TAS-B_gain,1
4,BM25+RM3(fb=30)+TAS-B_gain,1


## Recall analysis per `fb_terms` on DL20

Repeat the same `recall_100` comparison for DL20.


In [ ]:
bm25_rm3_runs_20 = {"BM25": runs_20["BM25"]}

for fb in FB_VALUES:
    pipe = build_bm25_rm3(index, num_results=NUM_RESULTS, fb_terms=fb)
    bm25_rm3_runs_20[f"BM25+RM3(fb={fb})"] = run_and_cache(
        pipe,
        topics_20_run,
        CACHE_DIR / f"{CACHE_PREFIX}_bm25_rm3_fb{fb}_dl20.pkl",
    )

bm25_rm3_perquery_20_recall = perquery_comparison(
    bm25_rm3_runs_20,
    topics_20_run,
    qrels_20_run,
    metric="recall_100",
)

bm25_rm3_analysis_20_recall = add_gain_columns(
    bm25_rm3_perquery_20_recall,
    baseline="BM25",
    systems=[f"BM25+RM3(fb={fb})" for fb in FB_VALUES],
)
bm25_rm3_analysis_20_recall = add_query_features(bm25_rm3_analysis_20_recall)


In [ ]:
bm25_rm3_metric_cols_20 = [f"BM25+RM3(fb={fb})" for fb in FB_VALUES]
bm25_rm3_gain_cols_20 = [f"BM25+RM3(fb={fb})_gain" for fb in FB_VALUES]

bm25_rm3_analysis_20_recall[bm25_rm3_metric_cols_20].mean().sort_values(ascending=False)


In [ ]:
bm25_rm3_best_20 = best_system_per_query(bm25_rm3_analysis_20_recall, bm25_rm3_metric_cols_20)
summarize_best_systems(bm25_rm3_best_20)


In [ ]:
bm25_rm3_best_gain_20 = best_system_per_query(
    bm25_rm3_analysis_20_recall,
    bm25_rm3_gain_cols_20,
    output_column="best_gain_setting",
)
summarize_best_systems(bm25_rm3_best_gain_20, best_system_column="best_gain_setting")


In [ ]:
bm25_rm3_analysis_20_recall = label_gain(
    bm25_rm3_analysis_20_recall,
    gain_column=f"BM25+RM3(fb={FB_TERMS})_gain",
)

summarize_labels(bm25_rm3_analysis_20_recall, f"BM25+RM3(fb={FB_TERMS})_gain_label")


In [ ]:
top_queries(
    bm25_rm3_analysis_20_recall,
    f"BM25+RM3(fb={FB_TERMS})_gain",
    n=10,
    ascending=False,
)
